# LLM Prediction Evaluation

This notebook evaluates the predictive performance of Gemini 3.1 Flash lite using qualitative descriptions generated in the `llm-pipeline` notebook. We compare two approaches: Global Descriptions and Dimension Analysis, and benchmark them against the OLS numeric predictions.

## 1. Setup and Dependencies

In [ ]:
!pip install -q -U google-generativeai

import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive, userdata
import google.generativeai as genai
from datetime import datetime
from sklearn.metrics import mean_absolute_error, r2_score

# Mount Google Drive
drive.mount('/content/drive')

# Configure Gemini
GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
MODEL_NAME = 'gemini-3.1-flash-lite-latest'

# Constants
BASE_PATH = '/content/drive/MyDrive/numeric_inference_outputs/'
EVAL_DATA_FILE = os.path.join(BASE_PATH, 'top_significant_channels_eval.json')
LLM_ANALYSIS_FILE = os.path.join(BASE_PATH, 'llm_analysis_results.json')
OUTPUT_FILE = os.path.join(BASE_PATH, 'llm_evaluation_results.json')
CACHE_FILE = os.path.join(BASE_PATH, 'gemini_eval_cache.json')
EXPORT_GLOBAL_FILE = os.path.join(BASE_PATH, 'llm_global_results.json')
EXPORT_DIM_FILE = os.path.join(BASE_PATH, 'llm_dimension_results.json')


## 2. Load Data and Preprocess Statistics

In [ ]:
with open(EVAL_DATA_FILE, 'r') as f:
    eval_dataset = json.load(f)

with open(LLM_ANALYSIS_FILE, 'r') as f:
    llm_analysis = json.load(f)

def calculate_channel_stats(train_videos):
    log_views = np.log1p([v['actual_views'] for v in train_videos])
    return {
        'min': np.min(log_views),
        'max': np.max(log_views),
        'mean': np.mean(log_views),
        'q1': np.percentile(log_views, 25),
        'median': np.median(log_views),
        'q3': np.percentile(log_views, 75)
    }

channel_stats = {c['channel_id']: calculate_channel_stats(c['train_videos']) for c in eval_dataset}
print(f"Loaded {len(eval_dataset)} channels and LLM analysis.")

## 3. Utility Functions (Cache & Retry)

In [ ]:
def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_cache(cache):
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=4)

def get_gemini_completion(prompt, cache, key, retries=3, sleep_time=2):
    if key in cache:
        return cache[key]

    model = genai.GenerativeModel(MODEL_NAME)

    for i in range(retries):
        try:
            response = model.generate_content(prompt)
            text = response.text
            cache[key] = text
            save_cache(cache)
            time.sleep(sleep_time)
            return text
        except Exception as e:
            print(f"Error on attempt {i+1}: {e}")
            if i < retries - 1:
                time.sleep(sleep_time * (i + 1))
            else:
                raise e
    return None

def parse_batch_response(text, expected_count):
    """Parses a JSON list of numbers from the LLM response."""
    import re
    try:
        # Try finding a JSON-like list in the text
        match = re.search(r'\[[\s\d.,-]*\]', text)
        if match:
            nums = json.loads(match.group())
            return [float(n) for n in nums][:expected_count]
    except:
        pass
    
    # Fallback to simple number extraction if JSON fails
    numbers = re.findall(r"\b\d+(?:\.\d+)?\b", text)
    floats = [float(n) for n in numbers]
    if len(floats) > expected_count:
        # Heuristic: if we have indices + values, values are likely every second number
        if len(floats) == expected_count * 2:
            return floats[1::2]
        return floats[:expected_count]
    return floats

## 4. Prediction Engine: Global Predictions

Predicting views in batches of 10 using the global performance descriptions.

In [ ]:
cache = load_cache()
global_results = {}
export_global = {}

for channel in eval_dataset:
    cid = channel['channel_id']
    cname = channel['channel_name']
    stats = channel_stats[cid]
    description = llm_analysis['global_performance_descriptions'][cid]
    test_vids = channel['test_videos']
    
    print(f"--- Predicting Global for {cname} ---")
    predictions = []
    
    for i in range(0, len(test_vids), 10):
        batch = test_vids[i:i+10]
        titles = "\n".join([f"{j+1}. {v['title']}" for j, v in enumerate(batch)])
        
        prompt = f"""You are an expert YouTube performance analyst.
Channel: {cname}
Success Drivers Description:
{description}

Context (Logarithmic Views from Training Data):
- Min: {stats['min']:.2f}
- Max: {stats['max']:.2f}
- Average: {stats['mean']:.2f}
- Q1 (25th): {stats['q1']:.2f}
- Median: {stats['median']:.2f}
- Q3 (75th): {stats['q3']:.2f}

Task: Predict the logarithmic views for the following 10 video titles. 
Your predictions must be integers (round numbers) in the logarithmic view space.
Return ONLY a JSON list of {len(batch)} integers.

Titles:
{titles}
"""
        
        res_text = get_gemini_completion(prompt, cache, f"global_{cid}_batch_{i//10}")
        batch_preds = parse_batch_response(res_text, len(batch))
        
        if len(batch_preds) != len(batch):
            print(f"Batch length mismatch for {cname} batch {i//10}. Expected {len(batch)}, got {len(batch_preds)}.")
            print(f"Response: {res_text}")
            # Fill with None to keep indices aligned for global_results
            predictions.extend([None] * len(batch))
            continue
            
        predictions.extend(batch_preds)
        
        for j, v in enumerate(batch):
            export_global[v['title'].lower()] = {
                "video_id": v['video_id'],
                "predicted_log_views": batch_preds[j],
                "channel_id": cid,
                "channel_name": cname
            }
        
    global_results[cid] = predictions


## 5. Prediction Engine: Dimension Analysis Predictions

Predicting views in batches of 10 using significant dimensions and their PCA values.

In [ ]:
dim_results = {}
export_dim = {}

for channel in eval_dataset:
    cid = channel['channel_id']
    cname = channel['channel_name']
    stats = channel_stats[cid]
    dim_analysis = llm_analysis['channel_significant_dimension_analysis'][cid]
    test_vids = channel['test_videos']
    
    # Find which dimensions are significant for this channel
    p_values = np.array(channel['model']['p_values'][1:])
    sig_indices = np.argsort(p_values)[:5]
    
    print(f"--- Predicting Dimensions for {cname} ---")
    predictions = []
    
    for i in range(0, len(test_vids), 10):
        batch = test_vids[i:i+10]
        
        titles_with_pca = ""
        for j, v in enumerate(batch):
            pca_vals = [f"Dim {idx}: {v['reduced_embedding'][idx]:.3f}" for idx in sig_indices]
            titles_with_pca += f"{j+1}. {v['title']} ({', '.join(pca_vals)})\n"

        prompt = f"""You are an expert YouTube performance analyst using semantic latent dimensions.
Channel: {cname}
Dimension Analysis for this channel:
{dim_analysis}

Context (Logarithmic Views from Training Data):
- Min: {stats['min']:.2f}
- Max: {stats['max']:.2f}
- Average: {stats['mean']:.2f}
- Q1 (25th): {stats['q1']:.2f}
- Median: {stats['median']:.2f}
- Q3 (75th): {stats['q3']:.2f}

Task: Predict the logarithmic views for the following 10 video titles based on their specific dimension scores.
Your predictions must be integers (round numbers) in the logarithmic view space.
Return ONLY a JSON list of {len(batch)} integers.

Titles and Dim Scores:
{titles_with_pca}
"""
        
        res_text = get_gemini_completion(prompt, cache, f"dim_{cid}_batch_{i//10}")
        batch_preds = parse_batch_response(res_text, len(batch))
        
        if len(batch_preds) != len(batch):
            print(f"Batch length mismatch for {cname} batch {i//10}. Expected {len(batch)}, got {len(batch_preds)}.")
            print(f"Response: {res_text}")
            predictions.extend([None] * len(batch))
            continue

        predictions.extend(batch_preds)
        
        for j, v in enumerate(batch):
            export_dim[v['title'].lower()] = {
                "video_id": v['video_id'],
                "predicted_log_views": batch_preds[j],
                "channel_id": cid,
                "channel_name": cname
            }
        
    dim_results[cid] = predictions


## 6. Evaluation and Comparative Analysis

In [ ]:
metrics = []

for channel in eval_dataset:
    cid = channel['channel_id']
    cname = channel['channel_name']
    test_vids = channel['test_videos']
    
    actual = np.log1p([v['actual_views'] for v in test_vids])
    numeric = np.log1p([v['predicted_views'] for v in test_vids])
    
    def clean_preds(preds, actual_len):
        p = np.array(preds[:actual_len])
        mask = np.array([x is not None for x in p])
        return p, mask

    llm_global_raw, mask_global = clean_preds(global_results[cid], len(actual))
    llm_dim_raw, mask_dim = clean_preds(dim_results[cid], len(actual))
    
    def get_metrics(act, pred, mask):
        act_filtered = act[mask].astype(float)
        pred_filtered = pred[mask].astype(float)
        if len(pred_filtered) == 0: return None, None
        return mean_absolute_error(act_filtered, pred_filtered), r2_score(act_filtered, pred_filtered)
    
    mae_num, r2_num = mean_absolute_error(actual, numeric), r2_score(actual, numeric)
    mae_glob, r2_glob = get_metrics(actual, llm_global_raw, mask_global)
    mae_dim, r2_dim = get_metrics(actual, llm_dim_raw, mask_dim)
    
    metrics.append({
        "channel": cname,
        "MAE_Numeric": mae_num, "R2_Numeric": r2_num,
        "MAE_LLM_Global": mae_glob, "R2_LLM_Global": r2_glob,
        "MAE_LLM_Dim": mae_dim, "R2_LLM_Dim": r2_dim
    })

df_metrics = pd.DataFrame(metrics)
display(df_metrics)

print("\n--- Mean Performance ---")
print(df_metrics.mean(numeric_only=True))


## 7. Visualizations

In [ ]:
# Comparison of MAE across methods
plt.figure(figsize=(12, 6))
df_plot = df_metrics.melt(id_vars="channel", value_vars=["MAE_Numeric", "MAE_LLM_Global", "MAE_LLM_Dim"], 
                          var_name="Method", value_name="MAE")
sns.barplot(data=df_plot, x="channel", y="MAE", hue="Method")
plt.xticks(rotation=45, ha='right')
plt.title("MAE Comparison: Numeric vs LLM (Global) vs LLM (Dimensions)")
plt.show()

# Scatter plot for a sample channel
sample_cid = eval_dataset[0]['channel_id']
sample_name = eval_dataset[0]['channel_name']
actual = np.log1p([v['actual_views'] for v in eval_dataset[0]['test_videos']])
pred_glob = global_results[sample_cid][:len(actual)]
pred_dim = dim_results[sample_cid][:len(actual)]

plt.figure(figsize=(8, 8))
plt.scatter(actual, pred_glob, alpha=0.5, label="Global LLM")
plt.scatter(actual, pred_dim, alpha=0.5, label="Dimension LLM", marker='x')
plt.plot([min(actual), max(actual)], [min(actual), max(actual)], 'r--', label="Ideal")
plt.xlabel("Actual Log Views")
plt.ylabel("Predicted Log Views")
plt.title(f"Actual vs Predicted: {sample_name}")
plt.legend()
plt.show()

## 8. Final Analysis and Interpretation

### Findings
1. **Global vs. Dimension Predictions**: Compare if giving the LLM raw PCA values and dimension descriptions improves its grasp of the semantic space compared to just high-level success drivers.
2. **LLM vs. Numeric (OLS)**: Evaluate if the qualitative approach can compete with or exceed the statistical baseline.
3. **Channel Sensitivity**: Identify which channels are easier for the LLM to 'understand' qualitatively.

### Conclusion
[Insert detailed interpretation based on results here]

## 9. Export Results

Exporting the results to JSON files for further analysis. We also print statistics about the exported datasets.

In [ ]:
def export_and_print_stats(data, filepath, name):
    with open(filepath, 'w') as f:
        json.dump(data, f, indent=4)
    
    total_videos = len(data)
    channels = {}
    for item in data.values():
        cname = item['channel_name']
        channels[cname] = channels.get(cname, 0) + 1
    
    print(f"--- {name} Statistics ---")
    print(f"Total videos: {total_videos}")
    print(f"Total channels: {len(channels)}")
    for cname, count in channels.items():
        print(f"- {cname}: {count} videos")
    print(f"Saved to: {filepath}\n")

export_and_print_stats(export_global, EXPORT_GLOBAL_FILE, "Global LLM")
export_and_print_stats(export_dim, EXPORT_DIM_FILE, "Dimension LLM")
